# Data Prepation 4: Preprocessing

### 1. Split data into train and test for RF
Only add column 'DAYS_SINCE_LAUNCH' to the dataset <br>
Columns like 'NORAD_CAT_ID', 'INTLDES', 'SATNAME' can be excluded from the model during the model creation process using ydf

In [49]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder

# read data from csv
debris_df = pd.read_csv('../Data/Combined.csv')
debris_df.head()

,NORAD_CAT_ID,INTLDES,SATNAME,OBJECT_TYPE,COUNTRY,SITE,LAUNCH,EPOCH,PERIAPSIS,APOAPSIS,...,INCLINATION,RA_OF_ASC_NODE,MEAN_ANOMALY,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,SEMIMAJOR_AXIS,PERIOD,LIFETIME
0,23,1959-009B,JUNO II R/B,ROCKET BODY,US,AFETR,1959-10-13,1982-01-13 15:21:06.439968,506.194,762.990,...,50.3043,233.6360,61.7000,16909.0,0.000982,0.000093,0.0,7012.727,97.406,10869 days
1,28,1960-002A,THOR ABLE R/B,ROCKET BODY,US,AFETR,1960-04-01,1987-09-12 00:12:35.555904,600.706,625.344,...,48.3748,236.4746,264.6079,46764.0,0.000080,0.000006,0.0,6991.160,96.957,11415 days
2,30,1960-003A,THOR ABLESTAR R/B,ROCKET BODY,US,AFETR,1960-04-13,1961-04-17 11:27:12.924288,287.445,456.763,...,51.2500,272.4964,155.8967,5670.0,0.000000,0.001202,0.0,6750.239,91.989,492 days
3,33,1960-003C,THOR ABLESTAR DEB,DEBRIS,US,AFETR,1960-04-13,1960-08-06 21:56:44.331935,285.231,618.180,...,51.2879,72.6448,30.5053,1770.0,0.000000,0.000019,0.0,6829.840,93.621,95 days
4,35,1960-005B,SL-3 R/B,ROCKET BODY,CIS,TTMTR,1960-05-15,1960-06-08 00:20:02.506559,302.454,323.852,...,64.8899,183.4791,307.4432,380.0,0.000000,0.001764,0.0,6691.288,90.787,63 days


In [ ]:
# convert Launch and Epoch to numeric timestamps
debris_df['LAUNCH'] = pd.to_datetime(debris_df['LAUNCH'])
debris_df['EPOCH'] = pd.to_datetime(debris_df['EPOCH'])

# convert Lifetime to number of days
debris_df["LIFETIME"] = pd.to_timedelta(debris_df["LIFETIME"]).dt.days

#### Add days since launch as feature

In [ ]:
# Add the time difference between Launch and Epoch to the dataframe
debris_df.insert(debris_df.columns.get_loc('EPOCH') + 1, 'DAYS_SINCE_LAUNCH', (debris_df['EPOCH'] - debris_df['LAUNCH']).dt.days)

In [51]:
# split the data into training and test sets (80% training, 20% test)
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(debris_df, test_size=0.2, random_state=42)

print("Training set size:", len(train_df))
print("Test set size:", len(test_df))

Training set size: 19579
Test set size: 4895


In [52]:
# save to csv
train_df.to_csv("../Data/Train.csv", index=False)
test_df.to_csv("../Data/Test.csv", index=False)
debris_df.to_csv("../Data/DebrisRF.csv", index=False)

### 2. Preprocess data for the MLP

#### Handle DateTime colums

In [ ]:
# read data from csv
debrisMLP_df = pd.read_csv('../Data/DebrisRF.csv')

# convert LAUNCH and EPOCH columns to datetime format
debrisMLP_df['LAUNCH'] = pd.to_datetime(debrisMLP_df['LAUNCH'])
debrisMLP_df['EPOCH'] = pd.to_datetime(debrisMLP_df['EPOCH'])

# extract year and month from Launch date and insert after Launch column
debrisMLP_df.insert(debrisMLP_df.columns.get_loc('LAUNCH') + 1, 'LAUNCH_YEAR', debrisMLP_df['LAUNCH'].dt.year)
debrisMLP_df.insert(debrisMLP_df.columns.get_loc('LAUNCH') + 2, 'LAUNCH_MONTH', debrisMLP_df['LAUNCH'].dt.month)

# extract year and month from Epoch date and insert after Epoch column
debrisMLP_df.insert(debrisMLP_df.columns.get_loc('EPOCH') + 1, 'EPOCH_YEAR', debrisMLP_df['EPOCH'].dt.year)
debrisMLP_df.insert(debrisMLP_df.columns.get_loc('EPOCH') + 2, 'EPOCH_MONTH', debrisMLP_df['EPOCH'].dt.month)

# drop original Launch and Epoch columns
debrisMLP_df.drop(columns=['LAUNCH', 'EPOCH'], inplace=True)

# save to csv
debrisMLP_df.to_csv("../Data/DebrisMLP.csv", index=False)

#### Train/Test split

In [54]:
debrisMLP_df.columns

Index(['NORAD_CAT_ID', 'INTLDES', 'SATNAME', 'OBJECT_TYPE', 'COUNTRY', 'SITE',
       'LAUNCH_YEAR', 'LAUNCH_MONTH', 'EPOCH_YEAR', 'EPOCH_MONTH',
       'DAYS_SINCE_LAUNCH', 'PERIAPSIS', 'APOAPSIS', 'ARG_OF_PERICENTER',
       'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE',
       'MEAN_ANOMALY', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT',
       'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD', 'LIFETIME'],
      dtype='object')

In [55]:
# define feature and label columns
feature_columns = ['OBJECT_TYPE', 'COUNTRY', 'SITE', 'LAUNCH_YEAR', 'LAUNCH_MONTH', 'EPOCH_YEAR', 'EPOCH_MONTH', 'DAYS_SINCE_LAUNCH', 'PERIAPSIS', 'APOAPSIS', 'ARG_OF_PERICENTER', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'MEAN_ANOMALY', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD']
label_column = 'LIFETIME'

x = debrisMLP_df[feature_columns]
y = debrisMLP_df[label_column]

In [56]:
# split x and y data into train and test
# 80% training, 20% testing
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [57]:
# show datatypes in x_train
print(x_train.dtypes)

OBJECT_TYPE           object
COUNTRY               object
SITE                  object
LAUNCH_YEAR            int32
LAUNCH_MONTH           int32
EPOCH_YEAR             int32
EPOCH_MONTH            int32
DAYS_SINCE_LAUNCH      int64
PERIAPSIS            float64
APOAPSIS             float64
ARG_OF_PERICENTER    float64
MEAN_MOTION          float64
ECCENTRICITY         float64
INCLINATION          float64
RA_OF_ASC_NODE       float64
MEAN_ANOMALY         float64
REV_AT_EPOCH         float64
BSTAR                float64
MEAN_MOTION_DOT      float64
MEAN_MOTION_DDOT     float64
SEMIMAJOR_AXIS       float64
PERIOD               float64
dtype: object


#### One Hot Encoding of categorical features

In [58]:
# encode non numerical columns using One Hot Encoding 
from sklearn.preprocessing import OneHotEncoder

categorical_cols = x_train.select_dtypes(include=["object"]).columns.tolist()

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
x_train_encoded = encoder.fit_transform(x_train[categorical_cols]) # only fit the encoder to training dataset to avoid data leakage
x_test_encoded = encoder.transform(x_test[categorical_cols])

# convert to dataframes
encoded_cols = encoder.get_feature_names_out(categorical_cols)
x_train_encoded = pd.DataFrame(x_train_encoded, columns=encoded_cols, index=x_train.index)
x_test_encoded = pd.DataFrame(x_test_encoded, columns=encoded_cols, index=x_test.index)

# print columns 
print("Encoded categorical columns:")
for col in categorical_cols:
    print(f"- {col}")

Encoded categorical columns:
- OBJECT_TYPE
- COUNTRY
- SITE


#### Normlaizing numerical features using standardization

In [59]:
from sklearn.preprocessing import StandardScaler # (z-Normalisierung)
import numpy as np

# get only numeric columns
numeric_cols = x_train.select_dtypes(include=[np.number]).columns.tolist()

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train[numeric_cols]) # only fit scaler on training set to avoid data leakage
x_test_scaled = scaler.transform(x_test[numeric_cols])

# convert to dataframes
x_train_scaled = pd.DataFrame(x_train_scaled, columns=numeric_cols, index=x_train.index)
x_test_scaled = pd.DataFrame(x_test_scaled, columns=numeric_cols, index=x_test.index)

# print columns
print("Scaled numeric columns:")
for col in numeric_cols:
    print(f"- {col}")

Scaled numeric columns:
- LAUNCH_YEAR
- LAUNCH_MONTH
- EPOCH_YEAR
- EPOCH_MONTH
- DAYS_SINCE_LAUNCH
- PERIAPSIS
- APOAPSIS
- ARG_OF_PERICENTER
- MEAN_MOTION
- ECCENTRICITY
- INCLINATION
- RA_OF_ASC_NODE
- MEAN_ANOMALY
- REV_AT_EPOCH
- BSTAR
- MEAN_MOTION_DOT
- MEAN_MOTION_DDOT
- SEMIMAJOR_AXIS
- PERIOD


In [60]:
# combinde encoded and normalized features
x_train_final = pd.concat([x_train_encoded, x_train_scaled], axis=1)
x_test_final = pd.concat([x_test_encoded, x_test_scaled], axis=1)

In [61]:
# save to csv
x_train_final["LIFETIME"] = y_train
x_test_final["LIFETIME"] = y_test

x_train_final.to_csv("../Data/TrainPreprocessed.csv", index=False)
x_test_final.to_csv("../Data/TestPreprocessed.csv", index=False)